In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
import torch
import numpy as np
from sklearn.metrics import accuracy_score

In [2]:
df = pd.read_csv('cleaned_twitter_data.csv')
df = df.sample(1000000, random_state=42)

In [3]:
df.head()

,target,text,cleaned_text
592624,0,I swear IT has me stressing fucktons. I dunno ...,swear stress fuckton dunno whether debug websi...
37583,0,@Lis311 I can't do the deuce. it's waaayy to ...,cant deuc waaayi pack could even hear eachoth
1194395,1,@pattyk71163rn how about don't have a reason t...,dont reason pursu cop haha
176200,0,chili cheese fries a bad idea for lunch..,chili chees fri bad idea lunch
336915,0,@michaelnotte @siegertd @netlash Good luck wit...,good luck even bad miss session


In [4]:
texts = df['cleaned_text'].astype(str).tolist()
labels = df['target'].tolist()

In [5]:
len(texts)

1000000

In [6]:
len(labels)

1000000

In [7]:
train_texts, test_texts, train_labels, test_labels = train_test_split(texts, labels, test_size=0.2, random_state=42)

In [8]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

In [9]:
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)

In [10]:
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)

In [11]:
class TwitterDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    
    def __getitem__(self, index):
        item = {key: val[index] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[index])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TwitterDataset(train_encodings, train_labels)
test_dataset = TwitterDataset(test_encodings, test_labels)

In [12]:
len(train_dataset)

800000

In [13]:
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc}

In [16]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=100,
    fp16=True
)

In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [18]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.494072,0.459095,0.795610
2,0.451415,0.457422,0.803945


TrainOutput(global_step=200000, training_loss=0.4661813583374023, metrics={'train_runtime': 3520.9061, 'train_samples_per_second': 454.428, 'train_steps_per_second': 56.804, 'total_flos': 2.6907440352e+16, 'train_loss': 0.4661813583374023, 'epoch': 2.0})

In [19]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.45742249488830566, 'eval_accuracy': 0.803945, 'eval_runtime': 75.2279, 'eval_samples_per_second': 2658.59, 'eval_steps_per_second': 332.324, 'epoch': 2.0}


In [2]:
# check gpu available
import torch
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(device)

cuda
